# Test Suite: Data Loader & Schema Standardization
This notebook executes the unit tests and interactive validations for `src/data_loader.py`.
It tests:
- Primary cleaned dataset ingestion (`cleaned_reviews.csv`, He et al. 2022)
- Secondary generalization dataset ingestion (`fake reviews dataset.csv`, Salminen et al. 2022)
- Schema harmonization across the 12-column standardized schema
- Missing columns, null text checks, duplicate review IDs, and label validity (0 = Genuine, 1 = Fake)


In [1]:
import os
import sys
import unittest
import pandas as pd

# Ensure project root is in sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

print(f"Working Directory: {os.getcwd()}")
from src.data_loader import (
    load_cleaned_primary_dataset,
    load_salminen_dataset,
    validate_dataset_schema,
    UNIFIED_COLUMNS
)
print("Data Loader module imported successfully!")


Working Directory: e:\ML\mini project 2
Data Loader module imported successfully!


## Interactive Demonstration 1: Primary Cleaned Dataset (He et al., 2022)
Load a sample of 100 rows from `dataset/cleaned_reviews.csv` and inspect columns, data types, and labels.


In [2]:
df_primary = load_cleaned_primary_dataset(sample_size=100)
print(f"Loaded {len(df_primary)} rows from primary dataset.")
print(f"Columns: {list(df_primary.columns)}")
print("\nLabel distribution:")
print(df_primary["label"].value_counts())
display(df_primary.head(3))


Loaded 100 rows from primary dataset.
Columns: ['review_id', 'reviewer_id', 'product_id', 'product_category', 'review_title', 'review_text', 'rating', 'verified_purchase', 'timestamp', 'label', 'original_label', 'source_dataset']

Label distribution:
label
1    66
0    34
Name: count, dtype: int64


,review_id,reviewer_id,product_id,product_category,review_title,review_text,rating,verified_purchase,timestamp,label,original_label,source_dataset
0,R1PVNRYXFCXWHU,AHJDC3YN7C2ZXA7O77RJFQEYSYDQ,NaN,Amazon Products (Unspecified),Great Micro Fiber Mop,This is a great microfiber mop easily assemble...,NaN,NaN,NaN,1,1,He_et_al_Amazon_Cleaned
1,R1MLUO4NY0U6FW,AFRGKW4SZ3UCFM7PXE2CA6Y7GYKA,NaN,Amazon Products (Unspecified),VACATION,"It worked great, we all slept thru the night",NaN,NaN,NaN,0,0,He_et_al_Amazon_Cleaned
2,R3NNYLXOBR90DS,AFNCTXV2XJVUYGDOTBJVNKUMPLXA,NaN,Amazon Products (Unspecified),Help me a lot,I can't fall asleep in a quiet house. This t...,NaN,NaN,NaN,0,0,He_et_al_Amazon_Cleaned


## Interactive Demonstration 2: Secondary Dataset (Salminen et al., 2022)
Load a sample of 100 rows from `dataset/fake reviews dataset.csv` and verify `CG` (1) vs `OR` (0) binary mapping.


In [3]:
df_salminen = load_salminen_dataset(sample_size=100)
print(f"Loaded {len(df_salminen)} rows from Salminen et al. dataset.")
print(f"Source Dataset: {df_salminen['source_dataset'].iloc[0]}")
print("\nLabel distribution:")
print(df_salminen['label'].value_counts())
display(df_salminen.head(3))


Loaded 100 rows from Salminen et al. dataset.
Source Dataset: Salminen_Kaggle_Product_Reviews

Label distribution:
label
0    60
1    40
Name: count, dtype: int64


,review_id,reviewer_id,product_id,product_category,review_title,review_text,rating,verified_purchase,timestamp,label,original_label,source_dataset
0,SALMINEN_027957,NaN,NaN,Kindle_Store_5,,Eva is on her to find a way to escape her abus...,3.0,NaN,NaN,1,CG,Salminen_Kaggle_Product_Reviews
1,SALMINEN_039755,NaN,NaN,Clothing_Shoes_and_Jewelry_5,,This is my second Glamorise and I love the loo...,5.0,NaN,NaN,1,CG,Salminen_Kaggle_Product_Reviews
2,SALMINEN_024983,NaN,NaN,Kindle_Store_5,,I absolutely love this series. The Ravens are ...,4.0,NaN,NaN,0,OR,Salminen_Kaggle_Product_Reviews


## Interactive Demonstration 3: Automated Schema Validation
Run `validate_dataset_schema` to test for missing columns, empty texts, duplicates, and total row integrity.


In [4]:
report = validate_dataset_schema(df_primary)
print("Schema Validation Report:")
for k, v in report.items():
    print(f"  {k}: {v}")


Schema Validation Report:
  valid_schema: True
  missing_columns: []
  total_rows: 100
  missing_texts: 0
  empty_texts: 0
  duplicate_review_ids: 0
  label_distribution: {1: 66, 0: 34}


## Formal Unittest Suite Runner
Execute the complete test case class using Python's standard `unittest` framework.


In [5]:
class TestDataLoader(unittest.TestCase):
    
    def test_load_cleaned_primary_dataset_schema(self):
        df = load_cleaned_primary_dataset(sample_size=100)
        self.assertEqual(len(df), 100)
        for col in UNIFIED_COLUMNS:
            self.assertIn(col, df.columns)

    def test_load_salminen_dataset_schema(self):
        df = load_salminen_dataset(sample_size=100)
        self.assertEqual(len(df), 100)
        for col in UNIFIED_COLUMNS:
            self.assertIn(col, df.columns)
        self.assertEqual(df["source_dataset"].iloc[0], "Salminen_Kaggle_Product_Reviews")
        unique_labels = set(df["label"].unique())
        self.assertTrue(unique_labels.issubset({0, 1}))
            
    def test_schema_validation(self):
        df = load_cleaned_primary_dataset(sample_size=50)
        report = validate_dataset_schema(df)
        self.assertTrue(report["valid_schema"])
        self.assertEqual(len(report["missing_columns"]), 0)
        self.assertEqual(report["empty_texts"], 0)
        self.assertEqual(report["duplicate_review_ids"], 0)
        self.assertEqual(report["total_rows"], 50)
        
    def test_label_values(self):
        df = load_cleaned_primary_dataset(sample_size=50)
        unique_labels = set(df["label"].unique())
        self.assertTrue(unique_labels.issubset({0, 1}))

# Run unittest suite in notebook
suite = unittest.TestLoader().loadTestsFromTestCase(TestDataLoader)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
assert result.wasSuccessful(), "Some tests failed!"


test_label_values (__main__.TestDataLoader.test_label_values) ... ok
test_load_cleaned_primary_dataset_schema (__main__.TestDataLoader.test_load_cleaned_primary_dataset_schema) ... ok
test_load_salminen_dataset_schema (__main__.TestDataLoader.test_load_salminen_dataset_schema) ... ok
test_schema_validation (__main__.TestDataLoader.test_schema_validation) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.198s

OK
